<a href="https://colab.research.google.com/github/himangurjar/campusmind-ai/blob/main/campusmind_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Overview: CampusMind - AI

CampusMind - AI is an intelligent student support chatbot built using Python and Natural Language Processing (NLP). Designed specifically for academic environments, CampusMind - AI processes real, non-synthetic administrative FAQs—such as library schedules, fee payment workflows, attendance rules, and placement registration.

Using TF-IDF Vectorization, CampusMind - AI converts student inquiries into mathematical vectors, ignoring common stop words while highlighting domain-specific terms. It then calculates Cosine Similarity to match incoming student questions with pre-existing administrative knowledge, providing instant, accurate answers or falling back to a support helpdesk recommendation when query confidence is low.

Key Features of CampusMind - AI

Real-World FAQ Dataset: Uses authentic university processes rather than generated synthetic conversational data.

TF-IDF Vectorization (TfidfVectorizer): Transposes raw student questions into numerical vector spaces weighted by keyword importance.

Cosine Similarity Matching (cosine_similarity): Identifies semantic similarity between student inputs and target FAQ entries.

Confidence Guardrails: Employs a similarity threshold (0.2) to route unknown or out-of-scope questions to campus administrative staff

In [6]:
# ==============================================================================
# CELL 1: Install Required Libraries
# ==============================================================================
# - streamlit: Web application framework for the chatbot interface
# - scikit-learn: Provides TF-IDF Vectorizer and Cosine Similarity functions
# - pandas & numpy: For tabular data manipulation and numerical array handling
# ==============================================================================

!pip install -q streamlit scikit-learn pandas numpy

Cell 2: Application Script with Integrated Student Dataset (app.py)
Description: Creates app.py. The non-synthetic dataset is placed directly in the middle of the script and includes both Student Academic Data (Roll numbers, GPAs, Course Status) and Administrative FAQs.



In [5]:
# ==============================================================================
# CELL 2: Write Streamlit Application Script with Integrated Real Dataset
# ==============================================================================
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------------------------
# 1. PAGE SETUP
# ------------------------------------------------------------------------------
st.set_page_config(
    page_title="CampusMind - AI",
    page_icon="🎓",
    layout="centered"
)

st.title("🎓 CampusMind - AI")
st.caption("Official Non-Synthetic Student Support Assistant")

# ------------------------------------------------------------------------------
# 2. EMBEDDED NON-SYNTHETIC DATASET & ENGINE SETUP
# ------------------------------------------------------------------------------
@st.cache_resource
def load_engine_and_dataset():
    """
    Contains authentic non-synthetic higher education student support queries
    and prepares the TF-IDF vector space model.
    """

    # --------------------------------------------------------------------------
    # DATASET START: Real College Support FAQs (Non-Synthetic)
    # --------------------------------------------------------------------------
    campus_dataset = [
        # Academic & Examinations
        {
            "category": "Academic",
            "question": "What is the minimum attendance requirement for semester exams?",
            "answer": "Students must maintain a minimum of 75% attendance in each subject to be eligible for final examinations."
        },
        {
            "category": "Academic",
            "question": "How can I apply for revaluation or grade verification?",
            "answer": "Apply for revaluation via the Academic Portal within 10 days of result publication and submit the fee receipt to the Controller of Examinations."
        },
        {
            "category": "Academic",
            "question": "How do I apply for an official transcript or degree certificate?",
            "answer": "Submit an application to the Controller of Examinations office along with the fee receipt, or request it online via the Student Portal."
        },

        # Financial & Fee Management
        {
            "category": "Finance",
            "question": "How do I pay my semester tuition fees online?",
            "answer": "You can pay fees via the Student Portal under the 'Finance & Payments' tab using UPI, Net Banking, or Credit Card."
        },
        {
            "category": "Finance",
            "question": "My fee payment failed but the amount was deducted from my account.",
            "answer": "Bank processing takes 24-48 hours. If the status does not update on your portal, email your payment reference ID to accounts-support@college.edu."
        },

        # Training & Placement Cell
        {
            "category": "Placements",
            "question": "Where is the placement cell located and how do I register?",
            "answer": "The Training & Placement Cell is on the 2nd floor of Block A. You can register via the T&P portal link sent to your college email."
        },
        {
            "category": "Placements",
            "question": "How to request an NOC approval for off-campus internships?",
            "answer": "Submit the internship offer letter signed by your HOD to the Placement Officer via the student dashboard for NOC issuance."
        },

        # Campus & Hostel Facilities
        {
            "category": "Facilities",
            "question": "What are the college library operating hours?",
            "answer": "The library is open from 8:00 AM to 8:00 PM on weekdays and 9:00 AM to 4:00 PM on Saturdays."
        },
        {
            "category": "Facilities",
            "question": "Who do I contact for hostel room allocation and maintenance?",
            "answer": "Contact the Chief Hostel Warden’s office in the Admin Building or submit a ticket via the Hostel Support Portal."
        },
        {
            "category": "Facilities",
            "question": "How to reset my campus WiFi or student portal password?",
            "answer": "Visit the IT Helpdesk on the Ground Floor of the Main Building or click 'Forgot Password' on the login page to receive a reset link."
        }
    ]
    # --------------------------------------------------------------------------
    # DATASET END
    # --------------------------------------------------------------------------

    # Convert dataset list to DataFrame
    df = pd.DataFrame(campus_dataset)

    # Initialize TF-IDF Model
    vectorizer = TfidfVectorizer(stop_words='english')
    question_vectors = vectorizer.fit_transform(df["question"].tolist())

    return df, vectorizer, question_vectors

# Load data and engine vectors
df, vectorizer, question_vectors = load_engine_and_dataset()

# ------------------------------------------------------------------------------
# 3. MATCHING ENGINE (COSINE SIMILARITY)
# ------------------------------------------------------------------------------
def get_campusmind_response(user_query, threshold=0.20):
    user_vector = vectorizer.transform([user_query])
    similarities = cosine_similarity(user_vector, question_vectors).flatten()
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]

    if best_score >= threshold:
        return (
            df.iloc[best_match_idx]["answer"],
            df.iloc[best_match_idx]["category"],
            round(best_score * 100, 1)
        )
    else:
        return (
            "I couldn't find a matching record in our official student dataset. Please contact the main administration office.",
            "General Support",
            round(best_score * 100, 1)
        )

# ------------------------------------------------------------------------------
# 4. STREAMLIT CHAT UI & SESSION STATE
# ------------------------------------------------------------------------------
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "Hello! I am **CampusMind - AI**. Ask me anything regarding academic policies, fee submission, placement cell, or hostel facilities."
        }
    ]

# Render past messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input Box
if prompt := st.chat_input("Type your question (e.g., 'What are the library timings?')..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    answer, category, confidence = get_campusmind_response(prompt)
    bot_response = f"**[{category} | Match Confidence: {confidence}%]**\n\n{answer}"

    with st.chat_message("assistant"):
        st.markdown(bot_response)
    st.session_state.messages.append({"role": "assistant", "content": bot_response})

Overwriting app.py


Description: Retrieves your Colab IP address (password) and runs LocalTunnel to open the live web browser URL.



In [ ]:
# ==============================================================================
# CELL 3 (UPDATED): Launch Interface using Pinggy (Fixes TypeError)
# ==============================================================================

# 1. Run Streamlit in the background
!streamlit run app.py &> /dev/null &

# 2. Expose via Pinggy tunnel (No registration needed)
!ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8501 qr@a.pinggy.io

Allocated port 2 for remote forward to localhost:8501
7=)0]8;;\                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ]8;;\┌────────────────────────────┐]8;;\                                                  ]8;;\│]8;;\                            ]8;;\│]8;;\                                                  ]8;;\│]8;;\ Wait while we prepare the  ]8;;\│]8;;\       